In [ ]:
from http.server import SimpleHTTPRequestHandler
import http
import socketserver
import datetime
import pandas as pd
import json
import asyncio
import aiohttp



# Exercise 1


In [ ]:
class DataProvider:
    def __init__(self, data_file='dSST.csv'):
        """
        Initializes a DataProvider object with the specified data file.

        Parameters:
        - data_file (str): The path to the CSV data file (default: 'dSST.csv')
        """
        self.data = pd.read_csv(data_file) 

    def get_data(self, parameter='all'):
        """
        Retrieves data from the data source based on the specified parameter.

        Parameters:
        - parameter (None, int, list): Optional parameter to filter the data.
            - If None, returns all data.
            - If int, retrieves data for the specified year.
            - If list of length 2, retrieves data for the specified year range.

        Returns:
        - str: JSON representation of the retrieved data.

        Raises:
        - ValueError: If an invalid parameter is provided or if no data is found for the specified year/year range.
        """
        if parameter =='all':
            # Return all data as JSON
            return self.data.to_json( )

        if isinstance(parameter, int):
            # Retrieve data for the specified year
            year_data = self.data.loc[self.data['Year'] == parameter]
            if year_data.empty:
                raise ValueError("No data found for the specified year")
            return year_data.to_json( )

        if isinstance(parameter, list) and len(parameter) == 2:
            # Retrieve data for the specified year range
            start_year, end_year = parameter
            year_data = self.data.loc[(self.data['Year'] >= start_year) & (self.data['Year'] <= end_year)]
            if year_data.empty:
                raise ValueError("No data found for the specified year range")
            return year_data.to_json( )

        raise ValueError("Invalid parameter")


In [ ]:
# Testing the code
d = DataProvider()
print(d.get_data()) 
print(d.get_data(1991)) 
print(d.get_data([1991,2000])) 

In [ ]:
class CustomRequestHandler(SimpleHTTPRequestHandler):
    """
    Custom request handler for handling HTTP requests.
    """

    data_provider = DataProvider('dSST.csv')
        
    def do_GET(self):
        """
        Handle the GET request.
        """
        if self.path.startswith('/data'):
            self.handle_data_request()
        else:
            self.send_error(404, "Not found")

    def handle_data_request(self):
        """
        Handle the data request based on the request path.
        """
        Year = self.path[6:]

        try:
            '''Get the data for the given Year or range or 'all' and send the response'''
            if Year == 'all':
                data = self.data_provider.get_data('all')
            elif Year.isdigit():
                data = self.data_provider.get_data(int(Year))
            elif '-' in Year:
                from_Year, to_Year = map(int, Year.split('-'))
                data = self.data_provider.get_data([from_Year, to_Year])
            else:
                self.send_error(
                    400, 'Bad Request: Invalid Year or range format')
                return

            # send the response
            self.send_response(200)
            self.send_header('Content-type', 'application/json')
            self.end_headers()
            self.wfile.write(bytes(data, 'utf-8'))

        except ValueError as e:
            # send the error response
            self.send_error(400, 'Bad Request: ' + str(e))

 

In [5]:
PORT = 8080

# Create an object of the above class
http = socketserver.TCPServer(("", PORT), CustomRequestHandler)

# Start the server
print("serving at port", PORT)
http.serve_forever()
